# ZeRO руками: владельцы, коллективы и контрольная точка

**Вопрос.** Как ZeRO-1/2/3 уменьшают постоянную память, не меняя шаг оптимизатора? Сначала проведём четыре числа по кольцу all-reduce: его две половины и есть обмены ZeRO. Потом для четырёх параметров и двух рангов мы разложим состояние, воспроизведём `reduce-scatter` и `all-gather`, а затем сравним результат с неразделённым эталоном.

In [ ]:
import hashlib
import json
import platform

print({'python': platform.python_version(), 'lab': 'zero-ownership-v1'})
P = 7_000_000_000
WORLD = 8
GIB = 1024 ** 3

## 1. Память стадий

Считаем BF16 рабочие параметры и градиенты, FP32 главную копию и два момента AdamW. Промежуточные значения и временный полный слой сюда намеренно не входят.

In [ ]:
def persistent_bytes(parameters, world, stage):
    weights = 2 * parameters
    gradients = 2 * parameters
    optimizer = 12 * parameters  # FP32 master + m + v
    if stage == 0:
        return weights + gradients + optimizer
    if stage == 1:
        return weights + gradients + optimizer / world
    if stage == 2:
        return weights + (gradients + optimizer) / world
    if stage == 3:
        return (weights + gradients + optimizer) / world
    raise ValueError('stage must be 0..3')

memory = {stage: persistent_bytes(P, WORLD, stage) / GIB for stage in range(4)}
assert memory[0] > memory[1] > memory[2] > memory[3]
assert abs(memory[0] / memory[3] - WORLD) < 1e-12
memory

## 2. Кольцевой all-reduce из `labs/ch16`

Раздел главы «Лаборатория: от четырёх чисел до BookLM» и рисунок с бухгалтерами, которые передают числа по кольцу. В книге осталась короткая функция `ring_allreduce`. Полная версия лежит в `labs/ch16/ring_allreduce.py`: там же вторая реализация, которая проводит каждый фрагмент по кольцу по очереди и записывает его путь. При загрузке файл сам проверяет, что обе версии дают один ответ, что все 24 клетки таблицы рисунка совпали, что сложение в фазе all-gather считает вклад дважды и что кольца от одного до восьми рангов складывают случайные числа верно.

Файл загружается через `importlib` по явному пути, поэтому у него своё пространство имён. Путь ищется от текущего каталога вверх: ноутбук работает и из `notebooks/`, и из корня репозитория.

In [ ]:
import importlib.util
from pathlib import Path


def find_labs_ch16():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "labs" / "ch16"
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError("labs/ch16 не найден: запустите ноутбук внутри репозитория книги")


LABS_CH15 = find_labs_ch16()


def load_lab(name):
    lab_path = LABS_CH15 / f"{name}.py"
    spec = importlib.util.spec_from_file_location(f"ch16_{name}", lab_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)   # здесь выполняются все проверки файла
    return module


ring = load_lab("ring_allreduce")
print("labs:", LABS_CH15, "| файлов:", len(list(LABS_CH15.glob("*.py"))))

Теперь первый пункт лабораторного порядка из главы: после каждого шага печатаем, какой фрагмент получил каждый ранг и что в нём лежит. Шаги 1–3 — reduce-scatter, шаги 4–6 — all-gather. После шага 3 у каждого ранга ровно один готовый фрагмент: ранг $r$ владеет фрагментом $(r+1) \bmod p$. На этом месте ZeRO-2 и останавливается: ранг хранит только свою часть суммы градиента. Вторая половина кольца — это all-gather, которым ZeRO-3 собирает параметры перед слоем.

In [ ]:
history = []
ring_result = ring.ring_allreduce(ring.values, history)
p = len(ring.values)
totals = [sum(row[c] for row in ring.values) for c in range(p)]
assert ring_result == [totals] * p

for step in range(1, 2 * (p - 1) + 1):
    row = sorted((rank, chunk, value) for kind, s, rank, chunk, value in history if s == step)
    phase = "reduce-scatter" if step < p else "all-gather"
    print(step, phase, [f"ранг {r}: фрагм. {c} = {v}" for r, c, v in row])

# Владельцы после reduce-scatter: ранг r держит готовый фрагмент (r + 1) mod p.
owners_after_rs = {rank: chunk for kind, s, rank, chunk, value in history if s == p - 1}
assert owners_after_rs == {r: (r + 1) % p for r in range(p)}
assert all(value == totals[chunk] for kind, s, rank, chunk, value in history if s == p - 1)

# Сломанная вариация: прибавить вместо записи в фазе all-gather.
assert ring.broken_gather_value() == 3633 != totals[2]
{'owners_after_reduce_scatter': owners_after_rs, 'sent_per_rank': 2 * (p - 1)}

## 3. Reduce-scatter, локальный шаг и all-gather

Каждый ранг вычисляет локальный градиент всего вектора. `reduce-scatter` усредняет одинаковые позиции и оставляет владельцу только его половину. После локального обновления `all-gather` собирает новый полный вектор.

In [ ]:
weights = [1.0, 2.0, 3.0, 4.0]
local_gradients = [
    [0.2, 0.4, 0.6, 0.8],
    [0.4, 0.2, 1.0, 0.6],
]
learning_rate = 0.1

def mean_columns(rows):
    return [sum(values) / len(rows) for values in zip(*rows)]

def split_even(values, world):
    assert len(values) % world == 0
    width = len(values) // world
    return [values[i * width:(i + 1) * width] for i in range(world)]

global_gradient = mean_columns(local_gradients)
reference = [w - learning_rate * g for w, g in zip(weights, global_gradient)]
gradient_shards = split_even(global_gradient, 2)       # reduce-scatter result
weight_shards = split_even(weights, 2)                 # ZeRO-3 owners
updated_shards = [
    [w - learning_rate * g for w, g in zip(ws, gs)]
    for ws, gs in zip(weight_shards, gradient_shards)
]
gathered = [value for shard in updated_shards for value in shard]
assert gathered == reference
{'gradient_shards': gradient_shards, 'updated': gathered}

## 4. Жизненный цикл ZeRO-3

Полный параметр существует только между `all-gather` и освобождением. Трасса ниже намеренно хранит события, а не настоящие GPU-тензоры.

In [ ]:
def zero3_trace(shards):
    trace = []
    full = None
    trace.append(('persistent', tuple(tuple(x) for x in shards)))
    full = [value for shard in shards for value in shard]
    trace.append(('forward_all_gather', tuple(full)))
    trace.append(('forward_compute', sum(full)))
    full = None
    trace.append(('forward_free', full))
    full = [value for shard in shards for value in shard]
    trace.append(('backward_all_gather', tuple(full)))
    trace.append(('backward_compute', sum(full)))
    full = None
    trace.append(('backward_free', full))
    return trace

trace = zero3_trace(weight_shards)
assert trace[3] == ('forward_free', None)
assert trace[-1] == ('backward_free', None)
trace

## 5. Распределённая контрольная точка

Один файл ранга не является полной моделью. Манифест связывает все фрагменты с номером шага и запрещает публиковать неполный набор.

In [ ]:
def digest(values):
    payload = json.dumps(values, separators=(',', ':')).encode()
    return hashlib.sha256(payload).hexdigest()

rank_files = {
    rank: {'weights': shard, 'sha256': digest(shard)}
    for rank, shard in enumerate(updated_shards)
}
manifest = {'step': 1, 'world_size': 2, 'ranks': rank_files, 'complete': True}
assert set(manifest['ranks']) == set(range(manifest['world_size']))
reconstructed = [v for rank in range(2) for v in manifest['ranks'][rank]['weights']]
assert reconstructed == reference

broken = {'step': 1, 'world_size': 2, 'ranks': {0: rank_files[0]}, 'complete': False}
assert set(broken['ranks']) != set(range(broken['world_size']))
manifest

## Задания

1. Измените число рангов на четыре и напечатайте владельца каждого параметра.
2. Добавьте два состояния AdamW и докажите, что они никогда не собираются для прямого прохода.
3. Добавьте временный слой размером `L` и найдите пик `16P/p + 2L`.
4. Намеренно разделите локальные суммы градиентов дважды на `world_size`; поймайте расхождение с эталоном.
5. Смоделируйте сохранение, при котором ранг 1 завершился до публикации манифеста.
6. Перенесите тот же тест на DeepSpeed ZeRO-3 или PyTorch FSDP2 и сравните один обновлённый параметр.